In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 다음의 생성형 ai를 렝체인으로 구현하시오.
- 원하는 모델을 이용하여 나라를 입력하면 해당 나라의 가장 유명한 음식을 추천하는 렝체인을 구현(food_chain)
- 음식을 입력하면 레시피를 생성하는 렝체인을 구현(recipe_chain)
- 위의 두 체인을 연결하여, 나라이름을 입력받아, 해당 나라의 가장 유명한 음식의 레시피를 생성하는 렝체인 프로그램을 구현하시오.
# -
- llm 모델을 생성
- 음식 추천 invoke 구현
- 다양한 답변형식을 컨트롤할 수 있도록 음식 추천 구현
- 레시피 추천 invoke 구현
- 다양한 답변 형식을 컨트롤 할 수 있도록 레시피 생성형 ai 구현
- 위의 두 렝체인을 연결
- 최종 렝체인 구현
- 과제물에 적절한 파일 제출 여부
- 시간 엄수
- 적절한 모델 선택

In [37]:
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
import os
load_dotenv()

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4o-mini')

In [32]:
from langchain_core.prompts import PromptTemplate

# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(
    template = 'what is the famous food(dish) of {country}? Return a word', # {}안의 값을 새로운 값으로 대입 가능
    input_variable = ['country']
)
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(prompt_template.invoke({"country":"Japan"})))

text='what is the famous food(dish) of Japan? Return a word'


'Sushi'

In [41]:
# 프롬프트 템플릿 -> llm -> 출력파서를 연결하는 체인 생성
dish_chain = prompt_template | llm | output_parser
# 생성된 체인 invoke
dish_chain.invoke({"country":"Japan"})

'Sushi'

In [42]:
from langchain_core.output_parsers import StrOutputParser

# 명시적인 지시사항이 포함된 프롬프트
recipe_prompt = PromptTemplate(
    template = 'what is the recipe of {dish}? Return very simple process and in numbered korean', # {}안의 값을 새로운 값으로 대입 가능
    input_variable = ['dish']
)
# 프롬프트 템플릿에 값 주입
prompt = recipe_prompt.invoke({'dish':'Sushi'})

result = llm.invoke(prompt)

# 문자열 출력 파서를 이용하여 llm응답을 단순 문자열 변환
output_parser = StrOutputParser()
print('파서 결과 : ', output_parser.invoke(result))

파서 결과 :  물론입니다! 간단한 초밥 만드는 방법을 한국어로 번호를 매겨서 설명해 드리겠습니다.

1. **재료 준비하기**: 초밥용 쌀, 신선한 생선 혹은 채소, 적당한 식초, 설탕, 소금 준비하기.
2. **쌀 씻기**: 초밥용 쌀을 여러 번 씻어서 전분을 제거하기.
3. **쌀 반죽하기**: 씻은 쌀에 물을 더해 쌀이 부드러워질 때까지 끓이기.
4. **초밥 양념 만들기**: 식초, 설탕, 소금을 섞어 초밥 양념을 만들기.
5. **쌀에 양념 섞기**: 다 cooked 초밥용 쌀에 초밥 양념을 넣고 부드럽게 섞기.
6. **생선이나 채소 준비하기**: 좋아하는 생선이나 채소를 얇게 썰기.
7. **초밥 모양 만들기**: 양념된 쌀을 작은 덩어리로 눌러서 모양 만들기.
8. **토핑 올리기**: 준비한 생선이나 채소를 초밥 위에 얹기.
9. **서빙하기**: 완성된 초밥을 접시에 담아 서빙하기.

맛있게 드세요!


In [56]:
# 레서피 추측 체인 생성
recipe_chain = recipe_prompt | llm | output_parser

In [54]:
from langchain_core.output_parsers import StrOutputParser

# 명시적인 지시사항이 포함된 프롬프트
ingridient_prompt = PromptTemplate(
    template = 'what is the ingridient of {dish}? Return detail numbered korean list except recipe', # {}안의 값을 새로운 값으로 대입 가능
    input_variable = ['dish']
)
# 프롬프트 템플릿에 값 주입
prompt = ingridient_prompt.invoke({'dish':'Sushi'})

result = llm.invoke(prompt)


# 문자열 출력 파서를 이용하여 llm응답을 단순 문자열 변환
output_parser = StrOutputParser()
print('파서 결과 : ', output_parser.invoke(result))

파서 결과 :  Here’s a detailed numbered list of common ingredients used in sushi:

1. **밥 (Sushi Rice)**  
   - 미니멀한 밥: 쌀, 물, 식초, 설탕, 소금

2. **김 (Nori)**  
   - 해조류: 구운 해초

3. **생선 (Fish)**  
   - 연어: 신선한 연어
   - 참치: 신선한 참치
   - 기타: 광어, 조개, 새우 등

4. **야채 (Vegetables)**  
   - 오이: 얇게 썬 오이
   - 아보카도: 부드러운 아보카도
   - 당근: 얇게 채 썬 당근
   - 무: 무채 또는 무절임

5. **어패류 (Seafood)**  
   - 새우: 익힌 또는 생 새우
   - 조개: 굴, 맛조개 등
   - 연어알: 연어알(로이) 또는 다른 어란

6. **소스 (Sauces)**  
   - 간장: 다진 파나 고추를 곁들인 간장
   - 마요네즈: 스파이시 마요네즈
   - 유자 소스: 유자와 간장 혼합

7. **양념 (Seasonings)**  
   - 참기름: 고소한 맛을 위해
   - 고추가루: 매운 맛을 위해

8. **고명 (Garnishes)**  
   - 고추: 얇게 썬 고추
   - 깻잎: 장식용 또는 맛을 더하기 위해
   - 다진 파: 위에 뿌리기 위해

9. **기타 (Others)**  
   - 와사비: 생간장과 함께 제공되는 매운 양념
   - 생강: 초절임 생강 (가리)

이 리스트는 다양한 종류의 스시를 만드는데 필요한 기본적인 재료들입니다.


In [55]:
# 레서피 추측 체인 생성
ingridient_chain = ingridient_prompt | llm | output_parser

In [59]:
final_chain_r = {"dish" : dish_chain} | recipe_chain
final_chain_i = {"dish" : dish_chain} | ingridient_chain
print(final_chain_r.invoke({"country":"Japan"}))
print(final_chain_i.invoke({"country":"Japan"}))

여기 간단한 초밥 (Sushi) 레시피입니다. 

1. **재료 준비**: 초밥용 쌀, 물, 식초, 설탕, 소금, 원하는 생선(연어, 참치 등), 해조류(김) 및 채소(오이, 아보카도 등)을 준비합니다.

2. **쌀 씻기**: 초밥용 쌀을 흐르는 물에 여러 번 씻습니다.

3. **쌀 삶기**: 씻은 쌀과 물을 1:1.2 비율로 냄비에 넣고 끓입니다. 끓기 시작하면 불을 줄이고 뚜껑을 덮고 15분간 더 익힙니다.

4. **식초 혼합**: 식초, 설탕, 소금을 섞어 초밥용 소스를 만듭니다.

5. **쌀에 소스 섞기**: 쌀이 익으면 식초 혼합물을 넣고 부드럽게 섞어 식힙니다.

6. **초밥 만듦**: 손에 약간의 물을 묻히고 쌀을 한 덩이 만들어 원하는 생선이나 채소를 올립니다. 

7. **형태 잡기**: 모든 재료를 잘 감싸거나 성형하여 초밥 모양을 만듭니다.

8. **서빙**: 만들어진 초밥을 접시에 담고 즐깁니다!

이렇게 간단하게 초밥을 만들 수 있습니다!
Sure! Here’s a detailed list of common ingredients used in sushi, presented in a numbered format:

1. **밥 (Sushi Rice)**
   - 일본 쌀 (Japanese short-grain rice)
   - 식초 (Vinegar)
   - 설탕 (Sugar)
   - 소금 (Salt)

2. **해산물 (Seafood)**
   - 생선 (Fish)
     - 연어 (Salmon)
     - 참치 (Tuna)
     - 광어 (Flounder)
   - 조개류 (Shellfish)
     - 전복 (Abalone)
     - 새우 (Shrimp)
     - 게 (Crab)

3. **채소 (Vegetables)**
   - 오이 (Cucumber)
   - 아보카도 (Avocado)
   - 당근 (Carrot)
   - 파 (Green onion)

4. **해조류 (Seaweed)**
   -